# Generating single-sample networks with LIONESS

This notebook is used to create the single-sample networks (SSN) to be used in the training of GEA. The method is based on LIONESS and the resulting graphs are aggregated to the PPI network in order to have prior knowledge that can enrich each sample.

In [1]:
# Libraries
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from tqdm import tqdm
from scipy import sparse
import torch
import networkx as nx
from typing import Dict, List, Tuple
import joblib
from torch_geometric.data import Data, InMemoryDataset

# Select project / dataset
project_id = "PRJNA248469"

gene_data = pd.read_csv(f"data/IBD_Andre_data_filtered/{project_id}_matrix_filtered.tsv", sep = "\t")
metadata = pd.read_csv(f"data/IBD_Andre_data_filtered/{project_id}_metadata.txt", sep = ",")
ppi_network = pd.read_csv(f"data/IBD_Andre_data_filtered/{project_id}_ppi_network.tsv", sep = "\t")

## Gene co-expression matrix

First things first, we need to get the co-expression matrix for our whole dataset. To do so, we normalize the data first and after we compute the correlation for each pair of genes (both Pearson and Spearman will be used to compare final results of the whole pipeline).

In [2]:
# Get gene count data into format
gene_data_t = gene_data.rename(columns={"gene_symbol":"samples"}).set_index("samples").T

# Normalize data (log2 CPM)
lib_size = gene_data_t.sum(axis=1)
if (lib_size == 0).any():
    raise ValueError("One or more samples have zero library size (not possible)")
cpm = gene_data_t.div(lib_size, axis=0) * 1e6
norm_data = np.log2(cpm + 1.0)

# Filter data that is from undesired phenotype group
rel_metadata = metadata[["BioSample", "source_name"]].rename(columns={"BioSample":"sample", "source_name":"phenotype"})
joint_data = rel_metadata.merge(norm_data, left_on="sample", right_index=True, how="inner")

# for project PRJNA565216
# joint_data = joint_data[joint_data["phenotype"].isin(["Control_Ileum", "Ulcerative Colitis_Ileum", "Crohn's Disease_Ileum"])].reset_index(drop=True)

# for project PRJNA248469
joint_data = joint_data[joint_data["phenotype"].str.contains("IBD|CD|UC")].reset_index(drop=True)
joint_data["phenotype"] = joint_data["phenotype"].str.split(",").str[-1].str.strip()
joint_data = joint_data.set_index("sample")

pearson_corrs = []
spearman_corrs = []
grouped_data = []

# Compute correlation per phenotype group
for phenotype in joint_data["phenotype"].unique():
    # Select data from phenotype group
    group_data = joint_data[joint_data["phenotype"] == phenotype].drop(columns="phenotype")
    grouped_data.append(group_data)
    # Correlation matrix - Pearson
    pearson_corr = group_data.corr(method="pearson")
    pearson_corrs.append(pearson_corr)
    # Correlation matrix - Spearman
    spearman_corr = group_data.corr(method="spearman")
    spearman_corrs.append(spearman_corr)


## LIONESS - A theoretical overview

LIONESS starts with the network $G$ built on all $N$ samples. For each sample $s$, the network built on all samples except that one is $G_{-s}$. Then, the sample-specific network $G_s$ is:

$$
G_{s} = N * G - (N-1) * G_{-s}
$$

If $G$ and $G_{-s}$ are correlation matrices, directly applying this may produce values outside $[-1,1]$, which can complicate interpretation. To avoid this, we will transform coefficients $r$ to Fisher-z space $\mathrm{atanh}(z)$. Because $\mathrm{atanh()}$ is a bijection from (-1, 1) -> $\mathbb{R}$ and $\mathrm{tanh()}$ is the inverse, any linear combination you form in z-space is guaranteed to map back to a value inside (-1, 1). Then:

$$
Z_{s} = N * Z - (N-1) * Z_{-s}
$$

$$
G_{s} = \mathrm{tanh}(Z_{s})
$$

## Aggregating PPI network to LIONESS framework

We now add the PPI network to informed each network of already-known interactions between genes. By doing this we will add an extra filtering step to discard any edge that isn't present in the network. This, however, just allows us to use known co-expressions and doesn't give the possibility to find new ones. First, we extract the edges from it.

In [3]:
# Extract edges from PPI network
ppi_edges = set()
for _, row in ppi_network.iterrows():
    edge = tuple(sorted((row["preferredName_A"], row["preferredName_B"])))
    ppi_edges.add(edge)

print(f"A total of {len(ppi_edges)} unique interaction are present in the PPI network.")

A total of 7829 unique interaction are present in the PPI network.


Now that we have the edges we compute the LIONESS framework, retaining only the PPI edges.

In [4]:
sample_edges_pearson = {}
sample_edges_spearman = {}

for i, group_data in enumerate(grouped_data):
    # Relevant data
    samples = group_data.index.tolist()
    genes = group_data.columns.tolist()
    N = group_data.shape[0]
    threshold = 0.25

    # Correlations
    pearson_corr = pearson_corrs[i]
    spearman_corr = spearman_corrs[i]

    # Mapping from gene to index
    gene_to_idx = {gene: i for i, gene in enumerate(genes)}

    # Get indices for the edges that are in our PPI network
    ppi_indices = []
    ppi_gene_pairs = []
    for gene_a, gene_b in ppi_edges:
        if gene_a in gene_to_idx and gene_b in gene_to_idx:
            idx_a = gene_to_idx[gene_a]
            idx_b = gene_to_idx[gene_b]
            # Ensure i < j
            ppi_indices.append((min(idx_a, idx_b), max(idx_a,idx_b)))
            ppi_gene_pairs.append((gene_a, gene_b))

    # Fisher z of G
    G_pearson = pearson_corr.values
    G_spearman = spearman_corr.values
    Z_pearson = np.arctanh(np.clip(G_pearson, -1 + 1e-12, 1 - 1e-12)) # clipping to avoid arctanh(1) or arctanh(-1)
    Z_spearman = np.arctanh(np.clip(G_spearman, -1 + 1e-12, 1 - 1e-12))

    # Iterate on all samples, leave-one-out G_{-s} correlation and LIONESS in z-space
    for _, s in enumerate(tqdm(samples, desc="LIONESS samples (PPI filtered)")):
        # Drop s
        data_minus_s = group_data.drop(index=s)

        # Correlation on N-1 samples
        G_minus_s_pearson = data_minus_s.corr(method="pearson").values
        G_minus_s_spearman = data_minus_s.corr(method="spearman").values

        # Z-space
        Z_minus_s_pearson = np.arctanh(np.clip(G_minus_s_pearson, -1 + 1e-12, 1 - 1e-12))
        Z_minus_s_spearman = np.arctanh(np.clip(G_minus_s_spearman, -1 + 1e-12, 1 - 1e-12))

        # LIONESS
        Z_s_pearson = N * Z_pearson - (N - 1) * Z_minus_s_pearson
        Z_s_spearman = N * Z_spearman - (N - 1) * Z_minus_s_spearman

        # PPI filtering step: extract weights for just the PPI edges
        Z_s_ppi_pearson = np.array([Z_s_pearson[i,j] for (i,j) in ppi_indices])
        Z_s_ppi_spearman = np.array([Z_s_spearman[i,j] for (i,j) in ppi_indices])

        # Back to r-space
        G_s_ppi_pearson = np.tanh(Z_s_ppi_pearson)
        G_s_ppi_spearman = np.tanh(Z_s_ppi_spearman)

        # Filter correlations under threshold
        keep_mask_pearson = np.abs(G_s_ppi_pearson) >= threshold
        keep_mask_spearman = np.abs(G_s_ppi_spearman) >= threshold

        # DataFrame for kept edges
        kept_pairs_pearson = [ppi_gene_pairs[i] for i in np.where(keep_mask_pearson)[0]]
        kept_weights_pearson = G_s_ppi_pearson[keep_mask_pearson]
        edges_pearson = pd.DataFrame({
            "geneA": [p[0] for p in kept_pairs_pearson],
            "geneB": [p[1] for p in kept_pairs_pearson],
            "weight": kept_weights_pearson
        })
        sample_edges_pearson[s] = edges_pearson.reset_index(drop = True)

        kept_pairs_spearman = [ppi_gene_pairs[i] for i in np.where(keep_mask_spearman)[0]]
        kept_weights_spearman = G_s_ppi_spearman[keep_mask_spearman]
        edges_spearman = pd.DataFrame({
            "geneA": [p[0] for p in kept_pairs_spearman],
            "geneB": [p[1] for p in kept_pairs_spearman],
            "weight": kept_weights_spearman
        })
        sample_edges_spearman[s] = edges_spearman.reset_index(drop = True)

LIONESS samples (PPI filtered):   0%|          | 0/204 [00:00<?, ?it/s]

LIONESS samples (PPI filtered): 100%|██████████| 59/59 [00:14<00:00,  3.98it/s]


## Save into NetworkX and PyTorch Geometric objects

Now we have our training dataset, we are going to turned it into NetworkX graphs for easy introspection and PyTorch Geometric data for GNN training. First we define the functions that are going to be used for this.

In [5]:
# Convert {sample_id:pd.DataFrame(edges)} to dict of NetworkX graphs. Nodes include all genes present in gene_to_idx so graphs are consistent.
def sample_edges_to_nx(sample_edges: Dict[str, pd.DataFrame],
                       gene_to_idx: Dict[str, int],
                       directed: bool = True
                       ) -> Dict[str, nx.Graph]:
    graphs = {}
    # base graph will all nodes
    for s, df in tqdm(sample_edges.items(), desc="Building NetworkX objects"):
        if directed:
            G = nx.DiGraph()

        else:
            G = nx.Graph()

        # add all nodes
        G.add_nodes_from(gene_to_idx.keys())
        
        # add edges from the graph s
        for _, row in df.iterrows():
            a = row["geneA"]
            b = row["geneB"]
            w = float(row["weight"])
            # skip edges that can't be indexed
            if a not in gene_to_idx or b not in gene_to_idx:
                continue
            G.add_edge(a, b, weight=w)
        
        # add graph to dictionary
        graphs[s] = G

    return graphs

# Build PyG data objects. Node features x are the expression values for that sample.
def sample_edges_to_pyg(sample_edges: Dict[str, pd.DataFrame],
                        norm_data: pd.DataFrame,
                        genes: List[str],
                        gene_to_idx: Dict[str, int],
                        keep_all_nodes: bool = True,
                        directed: bool = True,
                        ) -> Tuple[List, Dict[str,int]]:
    data_list = []
    sample_to_idx = {}
    num_nodes = len(genes)

    # node-feature matrix for each sample
    for idx_s, (s,df) in tqdm(enumerate(sample_edges.items()), desc="Building PyG objects"):
        # expression vector for sample s
        if s not in norm_data.index:
            raise KeyError(f"Sample {s} not found in expression data index.")
        expr = norm_data.loc[s, genes].values.astype(np.float32).reshape(-1,1)
        x = torch.from_numpy(expr) # shape: [num_nodes, 1]

        # build edge_index and edge_attr only for edges present in df
        edges = []
        weights = []

        for _, row in df.iterrows():
            a = row["geneA"]
            b = row["geneB"]
            w = float(row["weight"])
            # skip edges that can't be indexed
            if a not in gene_to_idx or b not in gene_to_idx:
                continue
            
            # gene index 
            i = gene_to_idx[a]
            j = gene_to_idx[b]

            # PyG uses 0-indexed node ids (keep both directions for undirected graphs)
            edges.append([i, j])    
            weights.append(w)
            
            # for undirected graphs
            if directed == False:
                edges.append([j, i])
                weights.append(w)
            
        if len(edges) == 0:
            # create empty edge tensor
            edge_index = torch.empty((2,0), dtype=torch.long)
            edge_attr = torch.empty((0,1), dtype=torch.float32)
        
        else:
            # assign edges and weights
            edge_index = torch.tensor(edges, dtype = torch.long).t().contiguous()
            edge_attr = torch.tensor(weights, dtype=torch.float32).view(-1,1)
        
        # create graph data point
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
        # attach sample id 
        data.sample_name = s
        data_list.append(data)
        sample_to_idx[s] = idx_s
    
    return data_list, sample_to_idx

With this defined, now we can declare functions that are going to be used to save the PyG objects.

In [6]:
# Save edges into csv files
def save_edge_csv(sample_edges: Dict[str, pd.DataFrame],
                  outdir: str,
                  prefix: str="edges"
                  ):
    os.makedirs(outdir, exist_ok = True)
    for s, df in tqdm(sample_edges.items(), desc="Saving edges into csv files"):
        path = os.path.join(outdir, f"{prefix}_{s}.csv")
        df.to_csv(path, index=False)

# Save sparse adjacency matrix per sample with shape (num_genes, num_genes).
def save_sparse_adj(sample_edges: Dict[str, pd.DataFrame],
                    genes: List[str],
                    gene_to_idx: Dict[str,int],
                    outdir: str,
                    prefix: str="adj"
                    ):
    os.makedirs(outdir, exist_ok = True)
    n = len(genes)
    for s, df in tqdm(sample_edges.items(), desc="Saving adjacency matrices into files"):
        rows = []
        cols = []
        vals = []
        for _, row in df.iterrows():
            a = row["geneA"]
            b = row["geneB"]
            w = float(row["weight"])
            # skip edges that can't be indexed
            if a not in gene_to_idx or b not in gene_to_idx:
                continue
            i = gene_to_idx[a]
            j = gene_to_idx[b]

            rows.extend([i, j])
            cols.extend([j, i])
            vals.extend([w, w])
        
        if len(rows) == 0:
            mat = sparse.csr_matrix((n, n), dtype=np.float32)

        else:
            mat = sparse.csr_matrix((vals, (rows, cols)), shape=(n,n))
        joblib.dump(mat, os.path.join(outdir, f"{prefix}_{s}.joblib"))

# Save PyG object list
def save_pyg(data_list: List,
             outpath: str):
    torch.save(data_list, outpath)

Lets save everything.

In [7]:
outdir = f"graph_data/{project_id}"

sample_edges_all = [sample_edges_pearson, sample_edges_spearman]
reg_methods = ["pearson", "spearman"]

for i, sample_edges in enumerate(sample_edges_all):
    # Save csv with edge list for all samples
    save_edge_csv(sample_edges, outdir=f"{outdir}/{reg_methods[i]}/edges")
    # Save adjancency matrices
    save_sparse_adj(sample_edges, genes, gene_to_idx, outdir=f"{outdir}/{reg_methods[i]}/adj")
    # Build and save PyG object 
    data_list, sample_to_idx = sample_edges_to_pyg(sample_edges, norm_data, genes, gene_to_idx)
    save_pyg(data_list, outpath=f"{outdir}/{reg_methods[i]}/pyg_data.pt")

Saving adjacency matrices into files: 100%|██████████| 303/303 [01:07<00:00,  4.46it/s]
Building PyG objects: 303it [01:07,  4.52it/s]
Saving adjacency matrices into files: 100%|██████████| 303/303 [01:08<00:00,  4.41it/s]
Building PyG objects: 303it [01:08,  4.44it/s]


# Generating single-sample networks with SWEET